In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pickle 

In [4]:
##Load the dataset

data = pd.read_csv("Churn_Modelling.csv")
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [6]:
##Preprocess the data
## Drop iirelevent columns

data = data.drop(['RowNumber','CustomerId', 'Surname'], axis=1)
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [9]:
##Encode categorical variables

label_encoder_gender= LabelEncoder()
data["Gender"]=label_encoder_gender.fit_transform(data['Gender'])
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,1,39,5,0.00,2,1,0,96270.64,0
9996,516,France,1,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,0,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,1,42,3,75075.31,2,1,0,92888.52,1


In [45]:
##ONE HOT ENCODING

from sklearn.preprocessing import OneHotEncoder
onehot_encoder=OneHotEncoder()
geo_encoder = onehot_encoder.fit_transform(data[["Geography"]])
geo_encoder

geo_encoder.toarray()

KeyError: "None of [Index(['Geography'], dtype='object')] are in the [columns]"

In [24]:
onehot_encoder.get_feature_names_out(['Geography'])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [28]:
geo_encoder_df = pd.DataFrame(geo_encoder.toarray(),columns=onehot_encoder.get_feature_names_out(['Geography']))
geo_encoder_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [31]:
##Combine all the Columns

data=pd.concat([data.drop('Geography', axis=1),geo_encoder_df], axis=1)
data.head()

KeyError: "['Geography'] not found in axis"

In [ ]:
## Save the encodeer and scaler  in pickle file

with open ('label_encoder_gender.pk1', 'wb') as file:
    pickle.dump(label_encoder_gender, file)
with open ('label_encoder_geo.pk1', 'wb') as file:
    pickle.dump(onehot_encoder, file)


In [34]:
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [38]:
## Divide the data set into independent and dependent features

X=data.drop("Exited", axis=1)
y=data['Exited']

##Split the data in training and the testing sets

##X_train, X_test, y_test=train_test_split(X,y,test_size=0.2,random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

##Scale these features

scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)



In [40]:
X_train

array([[ 0.35649971,  0.91324755, -0.6557859 , ...,  1.00150113,
        -0.57946723, -0.57638802],
       [-0.20389777,  0.91324755,  0.29493847, ..., -0.99850112,
         1.72572313, -0.57638802],
       [-0.96147213,  0.91324755, -1.41636539, ..., -0.99850112,
        -0.57946723,  1.73494238],
       ...,
       [ 0.86500853, -1.09499335, -0.08535128, ...,  1.00150113,
        -0.57946723, -0.57638802],
       [ 0.15932282,  0.91324755,  0.3900109 , ...,  1.00150113,
        -0.57946723, -0.57638802],
       [ 0.47065475,  0.91324755,  1.15059039, ..., -0.99850112,
         1.72572313, -0.57638802]])

In [42]:
with open ('scler.pk1', 'wb') as file:
    pickle.dump(scaler, file)

In [46]:
data

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,1,39,5,0.00,2,1,0,96270.64,0,1.0,0.0,0.0
9996,516,1,35,10,57369.61,1,1,1,101699.77,0,1.0,0.0,0.0
9997,709,0,36,7,0.00,1,0,1,42085.58,1,1.0,0.0,0.0
9998,772,1,42,3,75075.31,2,1,0,92888.52,1,0.0,1.0,0.0


ANN Implementation


In [49]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

In [50]:
X_train.shape[1]

12

In [ ]:
##Build Our ANN Model

model=Sequential(
    [
        Dense(64, activation='relu', input_shape=(X_train.shape[1],)) , ##HL1 Connected with the input layer
        Dense(32, activation='relu',) , ## HL2 
        Dense(1, activation ='sigmoid')  ## Output
    ]
)

In [56]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_2 (Dense)             (None, 64)                832       
                                                                 
 dense_3 (Dense)             (None, 32)                2080      
                                                                 
 dense_4 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [58]:
import tensorflow
opt=tensorflow.keras.optimizers.Adam(learning_rate=0.01)
loss=tensorflow.keras.losses.BinaryCrossentropy()
loss

In [61]:
##Compile the model

model.compile(optimizer='adam',loss="binary_crossentropy", metrics=['accuracy'])

In [64]:
##Set the Tensorboard
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
log_dir='logs/fit' + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback=TensorBoard(log_dir = log_dir,histogram_freq=1)

In [66]:
##Set up the Early Stopping 

early_stopping_callback= EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

In [68]:
##Training the model
history=model.fit(
    X_train,y_train,validation_data=(X_test,y_test),epochs=100,
    callbacks=[tensorflow_callback,early_stopping_callback]
)

Epoch 1/100
250/250 [==============================] - 1s 4ms/step - loss: 0.3128 - accuracy: 0.8705 - val_loss: 0.3435 - val_accuracy: 0.8645
Epoch 2/100
250/250 [==============================] - 1s 4ms/step - loss: 0.3105 - accuracy: 0.8701 - val_loss: 0.3407 - val_accuracy: 0.8650
Epoch 3/100
250/250 [==============================] - 1s 3ms/step - loss: 0.3095 - accuracy: 0.8739 - val_loss: 0.3400 - val_accuracy: 0.8670
Epoch 4/100
250/250 [==============================] - 1s 3ms/step - loss: 0.3078 - accuracy: 0.8717 - val_loss: 0.3428 - val_accuracy: 0.8630
Epoch 5/100
250/250 [==============================] - 1s 3ms/step - loss: 0.3058 - accuracy: 0.8734 - val_loss: 0.3407 - val_accuracy: 0.8615
Epoch 6/100
250/250 [==============================] - 1s 3ms/step - loss: 0.3067 - accuracy: 0.8696 - val_loss: 0.3434 - val_accuracy: 0.8615
Epoch 7/100
250/250 [==============================] - 1s 3ms/step - loss: 0.3048 - accuracy: 0.8730 - val_loss: 0.3440 - val_accuracy: 0.8610

In [70]:
model.save('model.h5')

In [3]:
##Load Tensorboard Extensions

%load_ext tensorboard

In [9]:
%tensorboard --logdir logs/fit
!kill 27572
%tensorboard --logdir logs/fit --port 6007

Reusing TensorBoard on port 6006 (pid 27572), started 6 days, 17:15:47 ago. (Use '!kill 27572' to kill it.)

'kill' is not recognized as an internal or external command,
operable program or batch file.


In [ ]:
##Load the pickle file



'kill' is not recognized as an internal or external command,
operable program or batch file.
